# Flan-T5 Base Model Fine-Tuning for Question-Answering
ref: https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/flan-t5-samsum-summarization.ipynb

In [ ]:
# install requirements (if needed)
!pip install -r 'requirements.txt'

In [1]:
import evaluate
import time
import numpy as np
import torch

import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")

from transformers import (
    AutoModelForSeq2SeqLM
    ,AutoTokenizer
    ,DataCollatorForSeq2Seq
    ,Seq2SeqTrainer
    ,Seq2SeqTrainingArguments
    ,BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model, PeftConfig, PeftModel, prepare_model_for_kbit_training

import warnings
warnings.simplefilter("ignore", UserWarning) # ignore user warning about use_reentrant method while fine-tuning

import multiprocessing

# check number of CPU cores
print(f'Currently have {multiprocessing.cpu_count()} CPU cores.')

# check if CUDA GPU is available on torch
print(f'Is CUDA available on torch? {torch.cuda.is_available()}.')

'NoneType' object has no attribute 'cadam32bit_grad_fp32'
Currently have 11 CPU cores.
Is CUDA available on torch? False.


[nltk_data] Downloading package punkt to /Users/dahliama/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/Users/dahliama/anaconda3/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


## Load pre-trained model & tokenizer

In [4]:
# specify device type to use to run LLMs (i.e. CPU, GPU/CUDA, etc)
device_type = "cuda:0"

# set quantization config
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True
        ,bnb_4bit_use_double_quant=True
        ,bnb_4bit_quant_type="nf4"
        ,bnb_4bit_compute_dtype=torch.bfloat16
    )

# load pre-trained model and tokenizer
model_name = "google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
    ,device_map = device_type
    ,quantization_config=bnb_config
    ,trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name ,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (when using 4-bit quantization)

# check model parameters & structure
model

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear4bit(in_features=768, out_features=768, bias=False)
              (k): Linear4bit(in_features=768, out_features=768, bias=False)
              (v): Linear4bit(in_features=768, out_features=768, bias=False)
              (o): Linear4bit(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear4bit(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear4bit(in_features=768, out_features=2048, bias=Fa

In [5]:
# Setup LoRA config
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

lora_alpha =  512
lora_dropout = 0.01
lora_rank = 256

peft_config = LoraConfig(
    lora_alpha=lora_alpha
    ,lora_dropout=lora_dropout
    ,r=lora_rank
    ,bias="none"
    ,task_type="SEQ_2_SEQ_LM"
    ,target_modules=['q', 'v']
)

peft_model = get_peft_model(model, peft_config)

## Load & preprocess data

In [2]:
from datasets import load_dataset

In [ ]:
pip install -U datasets

In [10]:
data_files = {'train':'train_augment_data.json'
              ,'val':'val_augment_data.json'
              ,'test':'test_augment_data.json'}

data = load_dataset("json", data_files=data_files)
data

Extracting data files: 100%|██████████| 3/3 [00:00<00:00, 24.44it/s]


Dataset json downloaded and prepared to /root/.cache/huggingface/datasets/json/default-a7d543d592cfcb7a/0.0.0/0f7e3662623656454fcd2b650f34e886a7db4b9104504885bd462096cc7a9f51. Subsequent calls will reuse this data.


100%|██████████| 3/3 [00:00<00:00, 977.39it/s]


DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 147276
    })
    val: Dataset({
        features: ['question', 'answer'],
        num_rows: 24546
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 24517
    })
})

Need to preprocess data by truncating and padding before fine-tuning model

In [11]:
from datasets import concatenate_datasets

In [12]:
# The maximum total input sequence length after tokenization.
# Sequences longer than this will be truncated, sequences shorter will be padded.
tokenized_inputs = concatenate_datasets([data["train"], data["val"], data["test"]]).map(lambda x: tokenizer(x["question"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_source_length = max([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Max source length: {max_source_length}")

# The maximum total sequence length for target text after tokenization.
# Sequences longer than this will be truncated, sequences shorter will be padded."
tokenized_targets = concatenate_datasets([data["train"], data["val"], data["test"]]).map(lambda x: tokenizer(x["answer"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_target_length = max([len(x) for x in tokenized_targets["input_ids"]])
print(f"Max target length: {max_target_length}")

/opt/conda/lib/python3.10/site-packages/datasets/table.py:1427: FutureWarning: promote has been superseded by promote_options='default'.
  [ x   x   x | x   x   x ]


Max source length: 130


Max target length: 512


In [13]:
def preprocess_function(sample, padding="max_length"):
    '''
    Purpose: to preprocess data by truncating or padding
    @params sample: data samples
    returns: tokenized dataset that are same lengths
    '''
    # add prefix to the input for t5
    inputs = ["Please answer this question: " + item for item in sample["question"]]

    # tokenize inputs
    model_inputs = tokenizer(inputs, max_length=max_source_length, padding=padding, truncation=True)

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(text_target=sample["answer"], max_length=max_target_length, padding=padding, truncation=True)

    # If we are padding here, replace all tokenizer.pad_token_id in the labels by -100 when we want to ignore
    # padding in the loss.
    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = data.map(preprocess_function, batched=True, remove_columns=["question", "answer"])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")

Keys of tokenized dataset: ['input_ids', 'attention_mask', 'labels']


## Check model response prior to fine-tuning

In [ ]:
# define function to format data to prompt instruction format
def get_model_response(model, question):
    '''
    Purpose: to generate an answer from model for a given question
    @params model: the loaded LLM model
    @params question: a question for model to answer (str)
    returns: an answer (str)
    '''
    input_ids = tokenizer(question, return_tensors="pt").input_ids.to(device_type)
    outputs = model.generate(input_ids, max_new_tokens=512, no_repeat_ngram_size=2)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return response[0]

In [ ]:
# check response of model response before it is fine-tuned
questions = data['train']['question'][0:5]

for q in questions:
    print(f'Question: {q}')
    q = f'{q}'
    ans = get_model_response(model, q)
    print(f'Answer: {ans}\n')

Question: What is the history of the Soft Coated Wheaten Terrier?
Answer: The Soft Coated Wheaten Terrier was a breed of dog that was first introduced in the United States in 1939. The dog was originally bred as 'Sweet Coat', but was later renamed Swift' in honor of the American dog pioneer, Charles S. Wheaton.

Question: Why are there many flea products available?
Answer: fleas are a common symptom of swine flu.

Question: What are the behavioral reasons for dogs eating cat poop?
Answer: a poop sucks

Question: Why should you continue to reward your dog with treats even after it has reached you?
Answer: It is a reward for good behavior.

Question: Are there other brand names for carprofen?
Answer: no



## Fine-tune model

In [14]:
# set evaluation metric
metric = evaluate.load("rouge")

# helper function to postprocess text
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 4) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return result

In [15]:
# ignore tokenizer pad token in the loss
label_pad_token_id = -100

# set up data collator to do padding for data inputs and labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=peft_model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)

In [ ]:
import wandb
wandb.login(key="<WANDB_API_KEY>") # log into wandb
%env WANDB_PROJECT=FLAN-T5-base-qlora-augment-gpulab

wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


env: WANDB_PROJECT=FLAN-T5-base-qlora-augment-gpulab


### Augment Experiment 0
All checkpoints reside in "./augment-experiments/google/flan-t5-base_full_finetune"

In [21]:
# define training arguments to fine-tune model
training_args = Seq2SeqTrainingArguments(
    output_dir=f'augment-experiments/{model_name}_qlora'
    ,num_train_epochs=10
    ,per_device_train_batch_size=32  # batch size per GPU for training
    ,per_device_eval_batch_size=16
    ,gradient_accumulation_steps=2
    ,gradient_checkpointing=True
    ,optim="paged_adamw_32bit"
    ,logging_steps=10               # log onto console ever 'x' steps
    ,evaluation_strategy="epoch"    # requires an eval_dataset to use
    ,save_strategy="epoch"          # save after every epoch
    ,learning_rate=2e-4
    ,weight_decay=0.001
    ,max_grad_norm=0.3
    ,warmup_ratio=0.03
    ,lr_scheduler_type="cosine"
    ,disable_tqdm=False
    ,report_to="wandb"
    ,seed=55
    ,predict_with_generate=True
    ,fp16=False
    ,push_to_hub=False
    ,load_best_model_at_end=False
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    compute_metrics=compute_metrics,
)

In [22]:
# Train/fine tune model
model.config.use_cache = False
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
0,1.922900,1.651507,26.462900,10.806400,21.740500,23.227200,20.000000
2,1.786100,1.436548,28.839400,13.531100,24.234400,25.704800,19.999959
4,1.656300,1.311828,29.662500,14.372400,25.118700,26.563700,19.999959
6,1.545000,1.235973,30.081400,14.868700,25.564700,27.022300,19.999959
8,1.519900,1.209356,30.161500,14.993000,25.661900,27.111400,20.000000
9,1.510900,1.207734,30.160300,14.995300,25.659000,27.107900,20.000000


TrainOutput(global_step=23010, training_loss=1.684184327968977, metrics={'train_runtime': 58461.742, 'train_samples_per_second': 25.192, 'train_steps_per_second': 0.394, 'total_flos': 3.0184089792823296e+17, 'train_loss': 1.684184327968977, 'epoch': 10.0})

In [20]:
wandb.finish()

eval/gen_len,▁
eval/loss,▁
eval/rouge1,▁
eval/rouge2,▁
eval/rougeL,▁
eval/rougeLsum,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███


## Push fine-tuned model to HuggingFace

In [3]:
from huggingface_hub import notebook_login

In [4]:
notebook_login()

In [5]:
from transformers import AutoModelForSeq2SeqLM

In [6]:
# load fine-tuned model
model_name = f'checkpoint-18412'

# load fine-tuned model
trained_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [8]:
# push fine-tuned model to HuggingFace Hub
trained_model.push_to_hub("dahlia25/flan-t5-base-dog-qlora-augmented")

/Users/dahliama/anaconda3/lib/python3.11/site-packages/transformers/integrations/peft.py:389: FutureWarning: The `active_adapter` method is deprecated and will be removed in a future version.
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/dahlia25/flan-t5-base-dog-qlora-augmented/commit/aca33159fe2096bc894ec943f30c520f96f615db', commit_message='Upload T5ForConditionalGeneration', commit_description='', oid='aca33159fe2096bc894ec943f30c520f96f615db', pr_url=None, pr_revision=None, pr_num=None)

## Run inference & evaluate model

In [9]:
from transformers import pipeline
import evaluate
from tqdm import tqdm
from rouge_score import rouge_scorer

In [10]:
# specify device type to use to run LLMs
device_type = "mps"  # use mps for MacOS

# load fine-tuned model and tokenizer
base_model_name = "google/flan-t5-base"
model_name = "dahlia25/flan-t5-base-dog-qlora-augmented"

trained_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
    ,device_map = device_type
    ,trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name ,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (when using 4-bit quantization)

### Load test dataset for evaluation

In [11]:
from datasets import load_dataset

In [12]:
data_files = {'test':'test_augment_data.json'}

data = load_dataset("json", data_files=data_files)
data

Extracting data files:   0%|          | 0/1 [00:00<?, ?it/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset json downloaded and prepared to /Users/dahliama/.cache/huggingface/datasets/json/default-f3ee9fe2a52d64a1/0.0.0/e347ab1c932092252e717ff3f949105a4dd28b27e842dd53157d2f72e276c2e4. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

DatasetDict({
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 24517
    })
})

### Evaluate model with ROUGE & BLEU metrics

In [13]:
def calculate_rouge4(pred, ref):
    '''
    Purpose: to calculate ROUGE-4 score for a single data point
    @params pred: a list of predictions made by model (str)
    @params ref: a list of references/actual answers (str)
    returns: a dict of evaluation results (dict)
    '''
    # get rouge-4 score; since not included in evaluate load method
    scorer = rouge_scorer.RougeScorer(['rouge4'], use_stemmer=True)
    scores = scorer.score(pred, ref)

    res = scores['rouge4'][2]  # get fmeasure (precision+recall) results
    
    return res

In [14]:
def calculate_rouge_bleu(preds, refs):
    '''
    Purpose: to calculate ROUGE & BLEU metrics for a set of data samples
    @params preds: a list of predictions made by model (list of str)
    @params refs: a list of references/actual answers (list of str)
    returns: a dict of evaluation results (dict)
    '''

    rouge_metric = evaluate.load("rouge")
    rouge_res = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    
    # calculate average rouge-4 score
    rouge4_res = []
    for i in range(len(preds)):
        pred = preds[i]
        ref = refs[i]
        r4 = calculate_rouge4(pred, ref)
        rouge4_res.append(r4)
        
    r4_avg = np.mean(rouge4_res)
    
    # put all results together
    res = rouge_res
    rouge_res['rouge4'] = r4_avg
    
    # calculate bleu scores
    bleu_metric = evaluate.load("bleu")
    bleu_res = bleu_metric.compute(predictions=preds, references=refs)
    
    for i in range(len(bleu_res['precisions'])):
        name_ix = i+1
        
        k = f'bleu{name_ix}'
        res[k] = bleu_res['precisions'][i]
        
    return res

In [41]:
# define function to format data to prompt instruction format
def get_model_response(model, question):
    '''
    Purpose: to generate an answer from model for a given question
    @params model: the loaded LLM model
    @params question: a question for model to answer (str)
    returns: an answer (str)
    '''
    input_ids = tokenizer(question, return_tensors="pt").input_ids.to(device_type)
    outputs = model.generate(input_ids
                             ,max_new_tokens=512
                             ,no_repeat_ngram_size=2
                            )
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return response[0]

In [42]:
n = 5000  # number of samples

questions = data['test']['question'][:n]
refs = data['test']['answer'][:n]  # actual answer

# get model predictions
preds = []
for i in tqdm(range(len(questions))):
    q = questions[i]
    ans = get_model_response(trained_model, q)
    preds.append(ans)

100%|██████████████████████████████████████████████████████████████████████| 5000/5000 [10:10:35<00:00,  7.33s/it]


In [44]:
# save generated predictions
import os
import json

with open(f'test_set_preds.json', 'w', encoding='utf-8') as f:
    json.dump(preds, f, ensure_ascii=False, indent=4)
f.close()

In [45]:
# calculate rouge & bleu metrics
eval_results = calculate_rouge_bleu(preds, refs)

print(f'----------\n Results: \n {eval_results}')

----------
 Results: 
 {'rouge1': 0.2574676548605559, 'rouge2': 0.09421925554928001, 'rougeL': 0.1766769091216968, 'rougeLsum': 0.1766580388154695, 'rouge4': 0.029468992730382712, 'bleu1': 0.16853096790791844, 'bleu2': 0.05928678441886138, 'bleu3': 0.02995212651306311, 'bleu4': 0.018250656246697344}


### Evaluate model with Jaro-Winkler similarity metric

In [46]:
from textdistance import jaro_winkler

In [48]:
# calculate jaro-winkler distance
jaros = []
for i in range(n):
    pred = preds[i]
    ref = refs[i]
    distance = jaro_winkler(pred, ref)
    jaros.append(distance)
    
print(f'The average jaro-winkler distance is: {np.mean(jaros)}')

The average jaro-winkler distance is: 0.6301675851744


### Evaluate model with Cosine Embedding distance metric

In [ ]:
from langchain.evaluation import load_evaluator
from langchain.evaluation import EmbeddingDistance
import os

os.environ['OPENAI_API_KEY'] = '<OPENAI_API_KEY>'

In [53]:
evaluator = load_evaluator("embedding_distance"
                           ,distance_metric = EmbeddingDistance.COSINE)

In [54]:
# calculate cosine embedding distance
cos_dist = []
for i in tqdm(range(n)):
    pred = preds[i]
    ref = refs[i]
    distance = evaluator.evaluate_strings(prediction=pred, reference=ref)
    cos_dist.append(distance['score'])
    
print(f'The average cosine embedding distance is: {np.mean(cos_dist)}')

  0%|                                                                                    | 0/5000 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|█████████████████████████████████████████████████████████████████████████| 5000/5000 [21:57<00:00,  3.80it/s]

The average cosine embedding distance is: 0.09326193637023596
